### Passo 2: o	Classifique com modelos supervisionados (AutoAI).

#### Instalação do Ambiente

In [16]:
%pip install "pymongo[srv]"

  Obtaining dependency information for pymongo[srv] from https://files.pythonhosted.org/packages/12/72/e58b9df862edbf238a1d71fa32749a6eaf30a3f60289602681351c29093a/pymongo-4.15.3-cp311-cp311-win_amd64.whl.metadata
  Obtaining dependency information for dnspython<3.0.0,>=1.16.0 from https://files.pythonhosted.org/packages/ba/5a/18ad964b0086c6e62e2e7500f7edc89e3faa45033c71c1893d34eed2b2de/dnspython-2.8.0-py3-none-any.whl.metadata
   ---------------------------------------- 0.0/331.1 kB ? eta -:--:--
   ----------- ---------------------------- 92.2/331.1 kB 5.5 MB/s eta 0:00:01
   ---------------------------------- ----- 286.7/331.1 kB 4.5 MB/s eta 0:00:01
   ---------------------------------------- 331.1/331.1 kB 3.5 MB/s eta 0:00:00
   ---------------------------------------- 0.0/859.2 kB ? eta -:--:--
   -------------- ------------------------ 327.7/859.2 kB 10.2 MB/s eta 0:00:01
   ---------------------------- ---------- 634.9/859.2 kB 10.1 MB/s eta 0:00:01
   ------------------------


[notice] A new release of pip is available: 23.2.1 -> 25.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
%pip install dotenv

In [ ]:
%pip install ibm-watsonx-ai
%pip install lale==0.8.3
%pip install scikit-learn==1.3.2
%pip install xgboost==2.0.3 
%pip install lightgbm==4.2.0 
%pip install snapml==1.14.*
%pip install autoai_libs
%pip install onnxconverter-common
%pip install flask

#### Importação das Bibliotecas

##### Importação das Blibiotecas Gerais

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from wordcloud import WordCloud
from sklearn.metrics import classification_report
import sys, os
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), "../..")))
import config

##### Importação das Bibliotecas do Watson AI

In [ ]:
from ibm_watsonx_ai import Credentials, APIClient
from ibm_watsonx_ai.experiment import AutoAI
from ibm_watsonx_ai.helpers import DataConnection, ContainerLocation
from ibm_watsonx_ai.deployment import WebService
import getpass

#### Estabelecendo Conexão com o Watson AI

In [ ]:
# Define onde está localizado o dataset no IBM Cloud
training_data_references = [
    DataConnection(
        data_asset_id = config.DATA_ASSET_ID
    ),
]

# Define onde os resultados do experimento (modelos, métricas) serão armazenados
training_result_reference = DataConnection(
    location=ContainerLocation(
        path=config.RESULT_PATH,
        model_location=config.MODEL_LOCATION,
        training_status=config.TRAINING_STATUS
    )
)

In [ ]:
# Metadados do experimento AutoAI (configurações da tarefa de classificação)
experiment_metadata = dict(
    prediction_type='binary', # Tipo de problema (binário)
    prediction_column='Engenharia Social?', # Nome da coluna alvo
    holdout_size=0.1, # Proporção dos dados reservados para teste
    scoring='accuracy', # Métrica principal
    csv_separator=',',
    random_state=33,
    max_number_of_estimators=2, # Número máximo de modelos testados
    training_data_references=training_data_references,
    training_result_reference=training_result_reference,
    deployment_url = config.DEPLOYMENT_URL,
    project_id = config.PROJECT_ID,
    positive_label=1,
    drop_duplicates=True,
    include_batched_ensemble_estimators=['BatchedTreeEnsembleClassifier(ExtraTreesClassifier)', 'BatchedTreeEnsembleClassifier(LGBMClassifier)', 'BatchedTreeEnsembleClassifier(RandomForestClassifier)', 'BatchedTreeEnsembleClassifier(SnapBoostingMachineClassifier)', 'BatchedTreeEnsembleClassifier(SnapRandomForestClassifier)', 'BatchedTreeEnsembleClassifier(XGBClassifier)'],
    classes=[0.0, 1.0],
    feature_selector_mode='auto'
)

In [ ]:
# Detecta número de CPUs disponíveis no ambiente
import os, ast
CPU_NUMBER = 4
if 'RUNTIME_HARDWARE_SPEC' in os.environ:
    CPU_NUMBER = int(ast.literal_eval(os.environ['RUNTIME_HARDWARE_SPEC'])['num_cpu'])

In [ ]:
# Cria credenciais com a API key para se conectar ao Watsonx.ai
api_key = config.API_KEY

credentials = Credentials(
    api_key=api_key,
    url="https://us-south.ml.cloud.ibm.com"
)

In [ ]:
# Cria cliente para manipular o Watson AI
client = APIClient(credentials)

# Define o projeto padrão onde os ativos estão armazenados
if 'space_id' in experiment_metadata:
    client.set.default_space(experiment_metadata['space_id'])
else:
    client.set.default_project(experiment_metadata['project_id'])

# Associa o cliente ao dataset carregado
training_data_references[0].set_client(client)

#### Executando o AutoAI para obter a Melhor Pipeline:

In [ ]:
# Inicializa o experimento AutoAI com as credenciais e o projeto
experiment = AutoAI(credentials=credentials, project_id=experiment_metadata["project_id"])

# Executa o experimento com as configurações definidas
pipeline_optimizer = experiment.runs.get_optimizer(metadata=experiment_metadata)

In [ ]:
# Gera um resumo com as pipelines testadas
df_summary = pipeline_optimizer.summary()

# Seleciona a pipeline com melhor desempenho
best_pipeline_name = df_summary.index[0]

# Recupera o pipeline treinado no formato do scikit-learn
sklearn_pipeline_model = pipeline_optimizer.get_pipeline(
    pipeline_name=best_pipeline_name,
    astype=AutoAI.PipelineTypes.SKLEARN
)

In [ ]:
# Garante que o cliente está setado no projeto correto novamente
client = APIClient(credentials=credentials)
client.set.default_project(experiment_metadata["project_id"])
training_data_references[0].set_client(client)

# Recupera os dados de teste (holdout) para avaliação
_, X_test, _, y_test = training_data_references[0].read(experiment_metadata=experiment_metadata, with_holdout_split=True, use_flight=True)

In [ ]:
# Avaliação do modelo com métrica de acurácia
from sklearn.metrics import accuracy_score
print("Accuracy Score:", accuracy_score(y_test.values.ravel(), sklearn_pipeline_model.predict(X_test.values)))

In [ ]:
# Relatório detalhado de métricas: precisão, recall e F1-score
print("\nClassification Report:")
print(classification_report(y_test, sklearn_pipeline_model.predict(X_test.values)))

### Passo 3: Extraia as palavras mais comuns em mensagens suspeitas.

In [ ]:
# Filtra os textos classificados como eng_social (rótulo = 1)
messages = X_test.loc[y_test.values.ravel() == 1, 'Conteudo'].astype(str)

# Junta todos os textos em uma única string
texto_social_eng = " ".join(messages.values)

# Gera uma nuvem de palavras com as palavras mais comuns em mensagens suspeitas
wordcloud = WordCloud(width=800, height=400, background_color='white').generate(texto_social_eng)

# Exibe a nuvem de palavras
plt.figure(figsize=(10, 5))
plt.imshow(wordcloud, interpolation='bilinear')
plt.axis("off")
plt.title("Palavras comuns em tentativas de Engenharia Social")
plt.show()